# Geospatial and Temporal workflows

The goal of this notebook is to explore workflows for processing geospatial and temporal data and incorporating those data into a model with FloPy. FloPy has built in tools to help process shapefile, raster, and timeseries data into formats that are compatible with MODFLOW.

This example is loosely based on the Lucerne Valley Hydrologic Model (Stamos et al., 2022).

![](lv_model.png)

The Lucerne Valley is in the Mojave Valley region of Southern CA. The primary uses of water support Agriculture and Domestic households. Recharge into the basin primarily comes from the San Bernardino Mountains to the south; however, some recharge also comes into the basin in the Granite mountains to the north. The groundwater basin also has a historic evaporative outlet at Lucerne Lake, which is an evaporation pan that used to support pheatophytes (salt tolerant plants).

In this example, we'll build a highly simplified model ("teaching version") of this groundwater basin using both geospatial and temporally varying data.

In [ ]:
import flopy
from flopy.discretization import StructuredGrid

import datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from pathlib import Path
import shutil

import warnings

warnings.simplefilter("ignore", DeprecationWarning)

In [ ]:
data_path = Path("../data/geospatial_and_temporal")

# study area and DEM
study_area = data_path / "study_area.shp"
dem_data = data_path / "lv_dem.tif"

# hydraulic conductivity and storage
hk_file = data_path / "borehole_k.shp"
sto_file = data_path / "borehole_sto.shp"

# model faults
faults = data_path / "faults.shp"

# lake boundary and climate data
climate_file = data_path / "historic_climate.csv"
lake_boundary = data_path / "Lucerne_lake.shp"

# agriculture
ag_areas = data_path / "lvhm_historic_ag.shp"
ag_wells = data_path / "lvhm_ag_well_locs.shp"
ag_withdrawal = data_path / "lvhm_ag_pump.csv"

# recharge location
rch_locs = data_path / "lvhm_recharge.shp"


## Setting up a base MFSimulation object

The first step in creating a new modflow 6 simulation is to instantiate a `MFSimulation` object. Once the `MFSimulation` object is created, a time discretization package, a solver package, and model objects can be added to the simulation.

In the block below create your simulation object, add a `ModflowTdis`, a `ModflowIms` and create a model object using `ModflowGwf`

For this model, we are going to simulate the historical conditions in the basin from 1941 through the end of 1959. In this simulation we'll use yearly stress periods and start with a single steady-state stress period to approximate pre-development conditions.

In [ ]:
# specify start date time for 1 second before 1941 using a date time string
start_time = "1940-12-31T23:59:59"

# setup the time record
time_record = []
for year in range(1941, 1960):
   time_record.append((365, 4, 1))

nper = len(time_record)
time_units = "days"
complexity = "MODERATE"
model_name = "lvhm_simple"
sim_ws = data_path / "lvhm_simple"

In [ ]:
# live code example


## Building a modelgrid and discretization package from geospatial data

FloPy includes a utility class named `Raster`. The `Raster` class contains methods for resampling raster data to model grids, cropping rasters, sampling raster data values in a profile along a line, and sampling points.

This example mimics the workflows presented in the previous `discretization_workflows.ipynb` to build an initial modelgrid and then uses that information to set up a MODFLOW DIS package object

In [ ]:
raster = flopy.utils.Raster.load(dem_data)

gdfaoi = gpd.read_file(study_area)
gdfaoi.head()

# inspect the raster and the grid boundary
fig, ax = plt.subplots(figsize=(8, 8))
raster.plot(ax=ax)
gdfaoi.plot(ax=ax, facecolor="None", edgecolor="red");

From the shapefile, we can get a bounding box of the study area which can be used to create model grid

In [ ]:
xmin, ymin, xmax, ymax = gdfaoi.total_bounds

In the code block below define a `dx`, `dy` (cell size), calculate `nrow` and `ncol` from the model boundaries and the cell sizes, and finally create `delr` and `delc` arrays. The DIS package documentation can be found [here](https://modflow6.readthedocs.io/en/6.2.0/_mf6io/gwf-dis.html) for reference.

In [ ]:
# Live code or class activity
dx = 500
dy = 500


In [ ]:
nlay = 3
xll = xmin
yll = ymin

### Creating an initial model grid instance

In the block below, create an initial model grid instance (it'll get deleted later). This model grid will be used to resample the DEM data to get the top array for the model and to intersect a shapefile that defines the active part of the model grid.

For a full representation of a model grid the following parameters are needed:

   - `delc`: row spacing in the column direction
   - `delr`: column spacing in the row direction
   - `top`: the model's top array. For the temporary grid any elevation can be provided as a 2d array of (nrow, ncol)
   - `botm`: the model's grid cell botm elevations. For the temporary grid this can be faked, but needs to be a 3d array with the dimensions (nlay, nrow, ncol)
   - `idomain`: the model's active and inactive extent. This can also be faked for the temporary grid. It'll be calculated later
   - `xoff`: the lower left corner of the model grid's x coordinate
   - `yoff`: the lower left corner of the model grid's y coordinate
   
Build a temporary model grid, called `fake_grid` in the block below. FloPy's `StructuredGrid()` object is used to create structured model grids.

In [ ]:
crs = gdfaoi.crs

# Live code or class activity, for simplicity use a single layer


### Intersecting vector data with a modelgrid

The `GridIntersect` utility class allows users to intersect vector data (shapefile) with modelgrids and will return information about the intersection. In this example the study area boundary is intersected with the modelgrid instance that was created in the previous block to create an `idomain` array for the model.

In [ ]:
# intersect and get the active cell id's
gix = flopy.utils.GridIntersect(fake_grid)

# live code, get active cells, where centroids intersect


Create an idomain array and then visualize it using FloPy's PlotMapView.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

pmv = flopy.plot.PlotMapView(modelgrid=fake_grid)
pmv.plot_array(idomain);

### Resampling raster data

The `Raster` class has a `resample_to_grid()` method that allows the user to resample raster data a model grid and/or perform geostatics on rasters. 

There are three parameters that can be supplied to `resample_to_grid()`:
   - `modelgrid`: a flopy.discretization.Grid instance
   - `band`: the raster band to sample
   - `method`: resampling method (more information about the methods can be found [here](https://flopy.readthedocs.io/en/latest/Notebooks/raster_intersection_example.html)

In [ ]:
# now we can create our dis file...
top = raster.resample_to_grid(
    fake_grid,
    band=raster.bands[0],
    method="min",
)
# we'll use average thicknesses from the model to create our layer elevations
botm = np.zeros((nlay, nrows, ncols))
botm[0] = top - 70 # meters, alluvial fill
botm[1] = botm[0] - 20 # meters, historic lake deposits
botm[2] = botm[1] - 90 # meters, historic mojave alluvial and fluvial deposits

### Create a DIS package

The `flopy.mf6.ModflowGwfdis()` package is used to create a structured rectalinier discretization package in FloPy. The preprocessing that has been done in the prior cells can be used to create this object. Parameters include:

   - `gwf`: the flopy model instance
   - `length_units`: the model length units
   - `xorigin`: the lower left corner x-coordinate
   - `yorigin`: the lower left corner y-coordinate
   - `nlay`: number of model layers
   - `nrow`: number of model rows
   - `ncol`: number of model columns
   - `delc`: row spacing in the column direction
   - `delr`: column spacing in the row direction
   - `top`: the model's top array. 
   - `botm`: the model's grid cell botm elevations.
   - `idomain`: the model's active and inactive extent.
   
Build a `flopy.mf6.ModflowGwfdis()` object

In [ ]:
length_units = "meters"

# live code



In [ ]:
# remove the temporary model grid instance...
try:
    del fake_grid
except NameError:
    pass

# and get a copy of the actual model grid instance
modelgrid = gwf.modelgrid
modelgrid.plot();

### Making a NPF package

The Node Property Flow package (NPF) stores hydraulic parameters for the model (e.g., horizontal hydraulic conductivity). Often this data (and storage parameters) are estimated from geologic properties that were recorded in borehole data or from pumping tests.

For this example, we'll import a point shapefile with hyraulic conductivity estimates at each point location. This data can then be interpolated to the model grid to produce a hydraulic conductivity fields for our model. To actually do the interpolation, we'll use a premade function that sets the data to our modelgrid.


In [ ]:
def interpolate_point_data(mg, gdf, data_column):
    """
    Method to interpolate point cloud data to the modelgrid

    :param mg: Flopy Grid object
    :param gdf: GeoDataFrame
    :param data_column: data column (values to interpolate)

    :return: np.array
    """

    import scipy.interpolate

    rxc, ryc = list(gdf.geometry.x), list(gdf.geometry.y)
    array = gdf[data_column].values

    xc, yc = mg.xcellcenters.ravel(), mg.ycellcenters.ravel()
    
    resampled_array = scipy.interpolate.griddata((rxc, ryc), array, (xc, yc), method="linear")

    # use nearest neighbor to fill any cells outside out the point cloud
    fill_array = scipy.interpolate.griddata((rxc, ryc), array, (xc, yc), method="nearest")
    resampled_array = np.where(np.isnan(resampled_array), fill_array, resampled_array)

    return resampled_array.reshape((mg.nrow, mg.ncol))
    

In [ ]:
# Load the borehole shapefile with conductivity values
gdfhk = gpd.read_file(hk_file)
gdfhk.head()

In [ ]:
# loop through the data columns and interpolate to a hk_array
hk_array = np.full(modelgrid.shape, 1e-10)

for ix, col in enumerate(["hk1", "hk2", "hk3"]):
    hk = interpolate_point_data(modelgrid, gdfhk, col)
    hk_array[ix] = hk

In [ ]:
# And plot the data to confirm it's been interpolated properly
fig, ax = plt.subplots(figsize=(8, 8))
pmv = flopy.plot.PlotMapView(modelgrid=modelgrid)
pc = pmv.plot_array(hk_array)
pmv.plot_inactive()
plt.colorbar(pc);

#### Build the node property flow package using `flopy.mf6.ModflowGwfnpf()`.

Four parameters are needed to build the node property flow package:
   - `gwf`: the flopy model object
   - `icelltype`: an array that defines if cells are confined or convertable. For this example, set all cells to confined storage
   - `k`: a horizontal conductivity array
   - `k33`: a vertical conductivity array. vk is unkown for this example however you could calculate a vk array using a vertical anisotropy factor...
   

In [ ]:
npf = flopy.mf6.ModflowGwfnpf(
    gwf,
    save_specific_discharge=True,
    icelltype=1,
    k=hk_array,
    k33=hk_array * 0.1
)


### Building a storage package

The storage package could be built in a similar way as the hydraulic conductivity package above, however for our example we'll be using constant values for SY and SS in this model.

In [ ]:
sto = flopy.mf6.ModflowGwfsto(
    gwf,
    ss=[0.00005, 0.00001, 0.00005],
    sy=[0.12, 0.2, 0.2],
    iconvert=[1, 1, 1],
    steady_state={0: True, 1: False},
    transient={0: False, 1: True}
)

## Initial Conditions

The basin initial condition is not well known, however we do know that there is limited exchange between the upper aquifer and the middle aquifer because of a lacustrine clay layer that acts as an aquitard between the two of them. For this model the intitial condition is set to the model top elevation.

In [ ]:
strt = np.zeros(modelgrid.shape)
strt[:] = modelgrid.top
print(np.min(strt), np.max(strt))

ic = flopy.mf6.ModflowGwfic(
    gwf,
    strt=strt*100
)

## Evapotranspiration from a historic dry lake bed.

USGS records from Mendenhall (1909) and Thompson (1929) describe a limited amount of natural discharge from the basin from a few springs in the western part of the basin and by evapotranspiration of phreatophytes near the dry Lucerne Lake bed (playa). Near the southwest part of the lake bed cottonwood trees were observed and sparse pheatophytic vegetation was also observed in other parts of the area.

Phreatophytes in the Mojave deset can grow extremely long tap roots and have large root systems. Mesquite trees have the longest tap roots that can be up to 25 meters deep in some cases. Other phreatophytes have shorter tap roots. For example, the creosote bush has a tap root that extends to a depth of about 1 meter, and cottonwood trees can have tap roots that extend to a depth of about 5 meters.


In [ ]:
gdf_lake = gpd.read_file(lake_boundary)
gdf_lake = gdf_lake.explode().reset_index(drop=True)

fig, ax = plt.subplots()
pmv = flopy.plot.PlotMapView(modelgrid=modelgrid, ax=ax)
pmv.plot_inactive()
pmv.plot_grid(lw=0.5)
gdf_lake.plot(ax=ax);

Use `flopy.utils.GridIntersect` to intersect Lucerne Lake with the modelgrid and identify which model cells to simulate evapotranspiration from. 

The process is similar to what was done to identify the active model extent earlier in the notebook.

In [ ]:
gix = flopy.utils.GridIntersect(modelgrid)

cellids = []
for geom in gdf_lake.geometry.values:
    result = gix.intersect(geom, contains_centroid=True)["cellids"]
    cellids.extend(result)

Now to load in historical evapotranspiration estimates for the basin which were derived from estimates produced by the Basin Characterization Model.

In [ ]:
climdf = pd.read_csv(climate_file)
climdf.head()

group data by year to create an annual flux for EVT package

In [ ]:
yearly = climdf.groupby(by="year", as_index=False)[["Eto_mm"]].sum()
yearly["Eto_mpd"] = (yearly["Eto_mm"] * 0.001) / 365.25
yearly["Eto_mpd"] *= 0.025 # use a low effective rate here due to very sparse coverage of plants
yearly["per"] = range(1, len(yearly) + 1)
yearly.head()

Using the cellids, create an evapotranspiration package. During this process we need to check that the cellid is within the active extent of the model.

The `flopy.mf6.ModflowGwfevt()` class is used to build an EVT6 package. The class documentation can be found [here](https://flopy.readthedocs.io/en/latest/source/flopy.mf6.modflow.mfgwfevt.html). The necessary parameters are:
   - `model`
   - `stress_period_data`: records of (cellid, surface, et_rate, et_depth)

In [ ]:
spd = {}
for per in range(nper):
    records = []
    if per == 0:
        et_rate = yearly["Eto_mpd"].mean()
    else:
        et_rate = yearly["Eto_mpd"].values[per - 1]

    for row, col in cellids:
        if modelgrid.idomain[0, row, col] > 0:
            records.append(((0, row, col), modelgrid.top[row, col], et_rate, 10))

    spd[per] = records

In [ ]:
# build the package
evt = flopy.mf6.ModflowGwfevt(
    gwf,
    stress_period_data=spd,
    pname="lake_evt"
)

## Simulating Recharge in the model

The Lucerne Valley is located in the Mojave desert and recieves very little precipitation on the valley floor. Most of the recharge to this basin comes from snowmelt in the San Bernardino mountains to the south and infiltrates into sediments that compose layer 3 south of the model boundary.

Prior recharge estimates from the Basin Characterization Model (Flint et. al., 2013) estimate about 680 acre-ft of recharge to the basin.

With this information, a recharge package can be built for the model.

In [ ]:
recharge = (680 * 1233.48) / 365.25  # acre-ft/yr to m3/day
rdf = gpd.read_file(rch_locs)

fig, ax = plt.subplots(figsize=(5, 5))
pmv = flopy.plot.PlotMapView(modelgrid=modelgrid, ax=ax)
pmv.plot_inactive()
rdf.plot(ax=ax);

Using `GridIntersect` identify the recharge cells

In [ ]:
gix = flopy.utils.GridIntersect(modelgrid)
result = gix.intersect(rdf.geometry.values[0])
# set the recharge to layer 3
cellids = []
for r, c in result["cellids"]:
    if modelgrid.idomain[2, r, c] > 0:
        cellids.append((2, r, c))

Build a recharge package using `flopy.mf6.ModflowGwfrch()`.  The class documentation can be found [here](https://flopy.readthedocs.io/en/latest/source/flopy.mf6.modflow.mfgwfrch.html). The necessary parameters are:
   - `model`
   - `maxbound`
   - `stress_period_data`: records of (cellid, recharge_rate)

**Note that recharge rate is in units of L/T (m3/day)**

In [ ]:
rch_rate = recharge / (len(cellids) * dx * dy) # convert from m3/day to m/d per cell
records = [(cid, rch_rate) for cid in cellids]
rch = flopy.mf6.ModflowGwfrch(
    gwf,
    stress_period_data={0: records},
    pname="sb_rch"
)


## Simulating model boundaries

Lucerne Valley is bounded on most sides by faults. The western part of the basin is bounded by the Helendale Fault system that runs southest to northwest and seperates the groundwater basin from the greater Mojave basin. In the north an unnamed fault runs east-west from the Granite mountains to the Ord mountains. In the southeast the Cougar Buttes Fault runs in a  southeast to northwest direction and separates the basin from the adjacent Johnson Valley aquifer system. The Cushenbury Fault, which is a splay off of the Helendale Fault, cuts across the southern part of the basin.

#### Note: with the exception of the Cushenbury Fault, these faults form the boundaries to this groundwater basin and can be represented as General Head boundaries
These faults have very low estimated conductivity. The Helendale fault and the unnamed northern fault conductivity is estimated to be around 1e-6. The Cougar Buttes fault provides slightly more exchange and has a conductivity between 1e-6 and 1e-5.

Heads across these faults are also higher than those in Lucerne Valley. Head on the west side of the Helendale fault is around 2900 ft asl, north of the unnamed fault is about 2890 ft asl and on the east side of the Cougar Buttes fault is about 2870 ft asl.


### Note2: the Cushenbury fault will be represented as a Horizontal flow barrier
The Cushenbury Fault zone has a hydraulic conductivity of about 1e-05

In this example, we'll build both a GHB and a HFB package

In [ ]:
gdff = gpd.read_file(faults)
gdff.head()

In [ ]:
# visualize the faults on the modelgrid
fig, ax = plt.subplots(figsize=(8, 8))
pmv = flopy.plot.PlotMapView(modelgrid=modelgrid, ax=ax)
pc = pmv.plot_ibound()
gdff.plot(ax=ax, color="red");

In [ ]:
# get the ghb faults and intersect with the model grid
ghb_df = gdff[gdff.fault_name != "cushenbury"]

ghb_dict = {}
gix = flopy.utils.GridIntersect(modelgrid)
for name, geom in zip(ghb_df["fault_name"], ghb_df["geometry"]):
    result = gix.intersect(geom)
    ghb_dict[name] = list(result.cellids)


#### Building the GHB package

Using the `ghb_cells` dictionary, we'll build a general head boundary condition package to represent faults on the edges of the model with `flopy.mf6.ModflowGwfghb()`. Head elevations and conductivity across the fault are as follows:
   - **helendale**: `k`=1e-06, `elev`=2900ft
   - **northern**: `k`=1e-06, `elev`=2890ft
   - **cougar**: `k`=3e-06`, `elev`=2870ft

Documentation for the `flopy.mf6.ModflowGwfghb()` class can be found [here](https://flopy.readthedocs.io/en/latest/source/flopy.mf6.modflow.mfgwfghb.html). Necessary input parameters are: 
   - `model`:
   - `stress_period_data`: records of (cellid, boundary_head, conductance)

In [ ]:
# build the stress period data
cell_thick = modelgrid.cell_thickness
fault_info = {
    "helendale": [1e-06, 883.92],
    "north": [1e-06, 880.872],
    "cougar": [3e-06, 874.776]
}

records = []
for name, cellids in ghb_dict.items():
    k, bhead = fault_info[name]
    for lay in range(nlay):
        for row, col in cellids:
            if modelgrid.idomain[lay, row, col] > 0:
                if bhead > modelgrid.botm[lay, row, col]:
                    thick = cell_thick[lay, row, col]
                    cond = (k * thick * dx)
                    records.append(((lay, row, col), bhead, cond))

now pass the record information to the GHB package constructor

In [ ]:
ghb = flopy.mf6.ModflowGwfghb(
    gwf,
    stress_period_data={0: records},
    pname="ghb_faults"
)

#### Building the HFB package

The final fault here can be represented as a horizontal flow barrier. For this we'll use the `make_hfb_array` utility in FloPy

In [ ]:
df_hfb = gdff[gdff.fault_name == "cushenbury"]
df_hfb.head()

The `make_hfb_array` utility takes linestring information and returns a recarray of hfb records for FloPy's HFB package object

In [ ]:
# live code this:
hydchr = 1e-05 * dx


now to build the HFB package

In [ ]:
# live code


### Pumping in the basin

Agriculture is the primary source of water withdrawals in the groundwater basin. For this example we are going to ignore domestic withdrawals and only simulate agricultural withdrawals. Due to water quality constraints here most pumping comes from the 3rd layer of the model (lower aquifer).

In [ ]:
# load our ag information
gdfag = gpd.read_file(ag_areas)
gdfag = gdfag.to_crs(epsg=26911)
gdfwel = gpd.read_file(ag_wells)
gdfwel = gdfwel.to_crs(epsg=26911)
dfpump = gpd.read_file(ag_withdrawal)

fig, ax = plt.subplots(figsize=(8, 8))
pmv = flopy.plot.PlotMapView(modelgrid=modelgrid, ax=ax)
pmv.plot_inactive()
gdfag.plot(ax=ax, facecolor="green")
gdfwel.plot(ax=ax, facecolor="None", edgecolor="k");

**What do we need to do with this data?**

   1. Locate cellids for each ag well (layer 3 only)
   2. Tie withdrawal data to model stress period (we can use the `ModelTime` class)
   3. Scale the annual withdrawal data based on irrigated acreage and apply pumping to wells

**Step 1, locate cellids for each ag well**

In [ ]:
# step one get cellids for each well
cellids = []
for geom in gdfwel.geometry.values:
    x, y = geom.x, geom.y
    cellid = modelgrid.intersect(x, y)
    cellids.append((2,) + cellid)
gdfwel["cellid"] = cellids
gdfwel.head()

**Step 2: tie withdrawal data to model stress period**

In [ ]:
modeltime = gwf.modeltime
pers = []
for year in dfpump["year"].values:
    dtobj = dt.datetime(int(year), 6, 1)
    per, stp = modeltime.intersect(dtobj)
    pers.append(per)
dfpump["per"] = pers
dfpump.head()

**Step 3: Create our scale factors and tie them to the wells**

In [ ]:
gdfag.head()

In [ ]:
total_ac = gdfag.area_ac.sum()
gdfag["pump_fac"] = gdfag["area_ac"] / total_ac

gdfwel = pd.merge(gdfwel, gdfag[["farm_id", "pump_fac"]], on="farm_id")
gdfwel.head()

**Finally we can make our stress period data and build the well package**

In [ ]:
spd = {}
for per, pumping in zip(dfpump["per"], dfpump["ag_withdrawal_acft"]):
    pumping = float(pumping)
    pumping *= ((-1 * 1233.48) / 365.25)  # acre-ft/year to cubic meters/day and change sign to -
    records = []
    for cellid, fac in zip(gdfwel["cellid"], gdfwel["pump_fac"]):
        rec = (cellid, pumping * fac)
        records.append(rec)
    spd[per] = records

In [ ]:
wel = flopy.mf6.ModflowGwfwel(
    gwf,
    stress_period_data=spd,
    pname="ag_pump"
)

## Create an output control package

In [ ]:
oc = flopy.mf6.ModflowGwfoc(
    gwf,
    saverecord={i: [("HEAD", "LAST"), ("BUDGET", "LAST")] for i in range(nper)},
    printrecord={i:[("BUDGET", "ALL")] for i in range(nper)},
    head_filerecord="lucerne_simple.hds",
    budget_filerecord="lucerne_simple.cbc",
    budgetcsv_filerecord="lucerne_simple.csv"
)

## Add an a observation package
If our study region hosts monitoring wells from which observations of groundwater head have been collected, we can set up an "observation" package directing MODFLOW to write the simulated heads to a dedicated file for whichever cells are listed.  

In [ ]:
obs_locs = []
obs_locs.append(("loc1", "head", (0, 9, 5)))
obs_locs.append(("loc2", "head", (2, 9, 5)))

obs = flopy.mf6.ModflowUtlobs(
    gwf,
    continuous={"lucerne_head_obs.csv": obs_locs},
    pname="head_obs"
)

## Write the simulation to file and run the simulation

In [ ]:
sim.write_simulation()
sim.run_simulation()

## Output visualization

## Load head results and plot

FloPy for modflow-6 has an `.output` attribute on the model object and many of the package objects. The `.output` attribute allows users to easily get head and budget information.

In [ ]:
# live coding
hds = gwf.output.head()
head = hds.get_alldata()[0]

### Plot the head results

FloPy includes plotting utilities that allow users to easily plot model data. `PlotMapView` can be used to plot arrays, shapefile data, discharge vectors, and boundary conditions. 

An example tutorial on `PlotMapView` can be found [here](https://flopy.readthedocs.io/en/latest/Notebooks/plot_map_view_example.html). Plot up the head results and the inactive area of the model grid.

In [ ]:
pmv = flopy.plot.PlotMapView(modelgrid=modelgrid, layer=2)
pc = pmv.plot_array(head)
pmv.plot_inactive()
plt.colorbar(pc);

In [ ]:
sim = flopy.mf6.MFSimulation.load(sim_ws=sim_ws)

In [ ]:
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(1, 1, 1, aspect="equal")
mapview = flopy.plot.PlotMapView(modelgrid=modelgrid)
ibound = mapview.plot_ibound()
mapview.plot_bc(package=wel, plotAll=True, color='green')
#mapview.plot_bc("RIV")
linecollection = mapview.plot_grid()

In [ ]:
fig = plt.figure(figsize=(15, 5))
ax = fig.add_subplot(1, 1, 1)

gw1 = sim.get_model()

# Next we create an instance of the PlotCrossSection class
xsect = flopy.plot.PlotCrossSection(model=gw1, line={"Column": 15})  #model=ml
linecollection = xsect.plot_grid()

patches = xsect.plot_ibound(color='white', color_noflow="grey", color_ch="orange")
patches = xsect.plot_bc("WEL", color="green")
pc = xsect.plot_array(head, head=head, alpha=0.5)
cb = plt.colorbar(pc, shrink=0.75)

t = ax.set_title("Column 16 Cross-Section - Model Grid")

In [ ]:
fig = plt.figure(figsize=(15, 5))
ax = fig.add_subplot(1, 1, 1)

gw1 = sim.get_model()

# Next we create an instance of the PlotCrossSection class
xsect = flopy.plot.PlotCrossSection(model=gw1, line={"Column": 15})  #model=ml
linecollection = xsect.plot_grid()

patches = xsect.plot_ibound(color='white', color_noflow="grey", color_ch="orange")
patches = xsect.plot_bc("WEL", color="green")
pc = xsect.plot_array(head, head=head, alpha=0.5)
cb = plt.colorbar(pc, shrink=0.75)

t = ax.set_title("Column 16 Cross-Section - Model Grid & Contours")

# contour array on top of heads
levels = np.arange(890, 896, 0.5)

# do black contour lines of the head array
contour_set = xsect.contour_array(head, head=head, levels=levels, colors="k")
plt.clabel(contour_set, fmt="%.1f", colors="k", fontsize=11)

### Plotting discharge vectors

`PlotCrossSection` has a `plot_vector()` method, which takes `qx`, `qy`, and `qz` vector arrays (ex. specific discharge or flow across a cell faces). The flow array values can be extracted from the cell by cell flow file using the `flopy.utils.CellBudgetFile` object as shown below.  Once they are extracted, they either be can be passed to the `plot_vector()` method or they can be post processed into specific discharge using  `postprocessing.get_specific_discharge`.  Note that `get_specific_discharge()` also takes the head array as an argument.  The head array is used by `get_specific_discharge()` to convert the volumetric flow in dimensions of $L^3/T$ to specific discharge in dimensions of $L/T$ and to plot the specific discharge in the center of each saturated cell. For this problem, there is no 'FLOW LOWER FACE' array since the Freyberg Model is a one layer model.

In [ ]:
cbb = gwf.output.budget()
spdis = cbb.get_data(text="SPDIS")[0]

qx, qy, qz = flopy.utils.postprocessing.get_specific_discharge(spdis, gw1, head=head)

In [ ]:
fig = plt.figure(figsize=(18, 5))
ax = fig.add_subplot(1, 1, 1)

ax.set_title("plot_array() and plot_vector()")
xsect = flopy.plot.PlotCrossSection(model=gw1, ax=ax, line={"Column": 15})
csa = xsect.plot_array(head, head=head, alpha=0.5)
patches = xsect.plot_ibound(head=head)
linecollection = xsect.plot_grid()
quiver = xsect.plot_vector(
    qx,
    qy,
    qz,
    head=head,
    hstep=2,
    normalize=True,
    color="orange",
    scale=40,
    headwidth=2,
    headlength=2,
    headaxislength=2,
    zorder=10,
)

cb = plt.colorbar(csa, shrink=0.75)


### A zone budget example

Use ZoneBudget.exe to extract specific budget items.

Let's remind ourselves where the EVT, RCH, and GHB boundaries are located.

In [ ]:
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(1, 1, 1, aspect="equal")
mapview = flopy.plot.PlotMapView(modelgrid=modelgrid)
ibound = mapview.plot_ibound()
mapview.plot_bc(package=evt, plotAll=True, color='blue', alpha=0.25)
mapview.plot_bc(package=rch, plotAll=True, color='cyan', alpha=0.25)
mapview.plot_bc(package=ghb, plotAll=True, color='limegreen', alpha=0.25)
linecollection = mapview.plot_grid()

Create a mask that spans only the upper half of the active model domain

In [ ]:
two_zn = idomain.copy()
two_zn[:, round(nrows/2):, :] = np.where(two_zn[:, round(nrows/2):, :] == 1, 2, 0)
plt.imshow(two_zn[0])
plt.colorbar()

In [ ]:
# Import the utility
#from flopy.utils import ZoneBudget
zb6_exe = "zbud6"
cpth = Path(sim_ws, "zbud6")
#if os.path.isdir(cpth):
if cpth.exists() and cpth.is_dir():
    shutil.rmtree(cpth)
    print("ZB directory deleted successfully.")
     
cpth.mkdir()

# now let's build a zonebudget model and run it!
zonbud = gw1.output.zonebudget(two_zn)
zonbud.change_model_ws(cpth)
zonbud.write_input()
success, buff = zonbud.run_model(exe_name=zb6_exe, silent=False)

### Getting the zonebudget output

We can then get the output as a recarray using the `.get_budget()` method or as a pandas dataframe using the `.get_dataframes()` method.


In [ ]:
zonbud.get_budget()

### That's hideous!

Instead, let's get the net flux using net=True flag to help summarize by zone

In [ ]:
zonbud.get_dataframes(net=True).iloc[0:20] # only display first 20 rows, or 2 stress periods

In [ ]:
# we can also pivot the data into a spreadsheet like format
zonbud.get_dataframes(net=True, pivot=True)

In [ ]:
# or get a volumetric budget by supplying modeltime
mt = gw1.modeltime

# budget recarray must be pivoted to get volumetric budget!
zonbud.get_volumetric_budget(mt, recarray=zonbud.get_budget(net=True, pivot=True))

## Plot simulated heads at monitoring well locations

In [ ]:
mon_wel = pd.read_csv(Path(sim_ws / "lucerne_head_obs.csv"))
mon_wel

In [ ]:
fig, ax = plt.subplots()
mon_wel["LOC1"].plot(ax=ax)
ax.set_title("Location 1 - Simulated Heads")
ax.set_xlabel("Time Step")
ax.set_ylabel("Head, m");

#### plotting the data with dates on the axes

The `modeltime` class that's attached to the `ModflowGwf` object allows us to tie model times to date-times and vice versa. Here's an example.

In [ ]:
modeltime = gwf.modeltime
dts = []
for time in mon_wel["time"].values:
    kper, kstp = modeltime.intersect(totim=time)
    dts.append(modeltime.get_datetime(kper, kstp))

In [ ]:
plt.plot(dts, mon_wel["LOC1"], "-b", label="Layer 1 head")
plt.plot(dts, mon_wel["LOC2"], "--k", label="Layer 3 head")
plt.ylabel("head, in meters")
plt.xlabel("date")
plt.legend();

### Clean-up

In [ ]:
if cpth.exists() and cpth.is_dir():
    shutil.rmtree(cpth)
    print("ZB directory deleted successfully.")
